In [1]:
# ===============================
# VALIDATION PIPELINE
# ===============================

import joblib
import numpy as np
import pandas as pd
from sklearn.metrics import r2_score, mean_squared_error
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

print("Validation script started...")

Validation script started...


In [5]:
print(df.columns)

Index(['country', 'year', 'biomass_relative_to', 'SST', 'Trawling_Hours',
       'country_encoded'],
      dtype='object')


In [13]:
# ===============================
# Load and Recreate Feature Engineering
# ===============================

import pandas as pd
import numpy as np

df = pd.read_csv('Final_fisheries_dataset.csv')

# Rename columns (same as training)
df = df.rename(columns={
    'country': 'Stock',
    'year': 'Year',
    'biomass_relative_to': 'Production'
})

df['Production'] = pd.to_numeric(df['Production'], errors='coerce')
df = df.dropna(subset=['Production']).sort_values(['Stock', 'Year'])

# FEATURE ENGINEERING (exact copy)

df['Prod_Lag1'] = df.groupby('Stock')['Production'].shift(1)
df['SST_Lag1'] = df.groupby('Stock')['SST'].shift(1)

df['Prod_3Y_Mean'] = df.groupby('Stock')['Production'].transform(lambda x: x.rolling(3).mean())

df['SST_Mean'] = df.groupby('Stock')['SST'].transform('mean')
df['SST_Anomaly'] = df['SST'] - df['SST_Mean']

df['Climate_Fishing_Pressure'] = df['SST'] * df['Trawling_Hours']

df = df.dropna()

# Final features used in model
features = [
    'Year',
    'SST',
    'Trawling_Hours',
    'Prod_Lag1',
    'SST_Lag1',
    'SST_Anomaly',
    'Climate_Fishing_Pressure'
]

X = df[features]
y = df['Production']

print("Feature engineering recreated successfully!")

Feature engineering recreated successfully!


In [14]:
# ===============================
# Recreate Time-Aware Split
# ===============================

train_mask = df['Year'] < 2013

X_train = X[train_mask]
X_test = X[~train_mask]

y_train = y[train_mask]
y_test = y[~train_mask]

print("Time-aware split recreated successfully!")

Time-aware split recreated successfully!


In [15]:
# ===============================
# Scale Test Data & Generate Predictions
# ===============================

# Scale only for Ridge
X_test_scaled = scaler.transform(X_test)

# Generate predictions
y_pred_ridge = ridge_model.predict(X_test_scaled)
y_pred_rf = rf_model.predict(X_test)

print("Predictions generated successfully!")

Predictions generated successfully!


In [16]:
# ===============================
# FINAL VALIDATION METRICS
# ===============================

from sklearn.metrics import r2_score, mean_squared_error
import numpy as np

# Ridge Metrics
ridge_r2 = r2_score(y_test, y_pred_ridge)
ridge_rmse = np.sqrt(mean_squared_error(y_test, y_pred_ridge))

# Random Forest Metrics
rf_r2 = r2_score(y_test, y_pred_rf)
rf_rmse = np.sqrt(mean_squared_error(y_test, y_pred_rf))

print("\n" + "="*50)
print("        VALIDATION RESULTS")
print("="*50)

print("\nRidge Regression")
print(f"R² Score : {ridge_r2:.4f}")
print(f"RMSE     : {ridge_rmse:.4f}")

print("\nRandom Forest")
print(f"R² Score : {rf_r2:.4f}")
print(f"RMSE     : {rf_rmse:.4f}")


        VALIDATION RESULTS

Ridge Regression
R² Score : 0.7112
RMSE     : 5350215.1890

Random Forest
R² Score : 0.6906
RMSE     : 5537381.0008


In [ ]:
#R² (Coefficient of Determination)

# Ridge: 71.12%

# Random Forest: 69.06%



In [19]:
# This means:

# Ridge explains slightly more variance in fish production compared to Random Forest.

# That is a strong result for an ecological time-series model.

In [20]:
# RMSE (Root Mean Squared Error)

# Lower is better.

# Ridge: 5.35 million

# RF: 5.53 million

In [ ]:
# ===============================
# FINAL VALIDATION RESULTS
# ===============================

# Ridge performs slightly better in prediction accuracy.

# Ridge Regression is the better performing model
# based on both:

# Higher R²

# Lower RMSE


In [22]:
# ===============================
# VALIDATION SUMMARY TABLE
# ===============================

import pandas as pd

validation_results = pd.DataFrame({
    "Model": ["Ridge Regression", "Random Forest"],
    "R2 Score": [ridge_r2, rf_r2],
    "RMSE": [ridge_rmse, rf_rmse]
})

print("\nValidation Summary:")
display(validation_results.round(4))


Validation Summary:


,Model,R2 Score,RMSE
0,Ridge Regression,0.7112,5.350215e+06
1,Random Forest,0.6906,5.537381e+06


In [24]:
# Here I will summarize the  steps I took to validate the models, the results and the final conclusion

# 1. Re-Analyzed Your Updated Training Files
# 2. Recreated the Exact Training Pipeline in Validation
# 3. Recreated the Exact Time-Aware Split
# 4. Loaded Saved Artifacts
# 5. Generated Predictions Properly
# 6. Calculated Final Validation Metrics
# 7. Model Comparison Conclusion